# Multi-region autumn phenology — exploratory audit

This notebook reads the relational release without forcing modalities into same-day rows. RCC90 events are operational camera metrics, not validated landscape peak-color dates.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TABLES = ROOT / 'outputs' / 'tables'
sites = pd.read_parquet(TABLES / 'sites.parquet')
coverage = pd.read_parquet(TABLES / 'coverage_audit.parquet')
obs = pd.read_parquet(TABLES / 'phenology_observations.parquet')
events = pd.read_parquet(TABLES / 'phenology_events.parquet')
inventory = pd.read_parquet(TABLES / 'dataset_inventory.parquet')
sites.shape, obs.shape, events.shape

## Autumn coverage from actual records

In [ ]:
matrix = coverage.pivot(index='site_id', columns='year', values='phenocam_usable_fraction')
fig, ax = plt.subplots(figsize=(15, 7))
image = ax.imshow(matrix, aspect='auto', vmin=0, vmax=1, cmap='viridis')
ax.set_yticks(range(len(matrix)), matrix.index)
ax.set_title('Usable Sep–Nov camera-day fraction')
fig.colorbar(image, ax=ax);

## Seasonal RCC90 trajectories remain platform-specific

In [ ]:
site_id, year = 'harvard_forest', 2020
subset = obs[(obs.site_id == site_id) & (obs.observation_type == 'rcc_90') & obs.quality_usable & (obs.timestamp.dt.year == year) & obs.timestamp.dt.month.isin([9,10,11])]
for platform_id, group in subset.groupby('platform_id'):
    plt.plot(group.timestamp, group.value, label=platform_id)
plt.ylabel('RCC90'); plt.legend(); plt.title(f'{site_id}, {year}');

## Within-site versus between-site variation

In [ ]:
events['event_doy'] = events.event_date.dt.day_of_year
site_year = events.groupby(['site_id','year'], as_index=False).event_doy.median()
by_site = site_year.groupby('site_id').event_doy.agg(['count','mean','std','min','max'])
display(by_site)
print('Median within-site SD:', by_site['std'].median())
print('Between-site SD of means:', by_site['mean'].std())

## Cross-modal readiness, without fabricated coverage

In [ ]:
inventory.groupby(['dataset_id','availability'], dropna=False).size().rename('site_entries').reset_index()